In [ ]:
# ========================================================================
# SUBIT-Ink v22.1 Generalization Fix — KAGGLE NOTEBOOK
# Vesuvius Challenge - Ink Detection
#
# v22.1 ЗМІНИ ВІД v22.0 (виправлення public/private gap):
#   1. Просторова крос-валідація: навч. F1+F2+F3(80%), вал. F3(20% по висоті).
#   2. Збалансований Tversky (α=0.5, β=0.5): Recall з ~0.40 → 0.50+.
#   3. Знижений PRECISION_W (0.05 замість 0.15): менший тиск на точність.
#   4. Більше даних: 4500 train-патчів, 25 епох.
#   5. Обмеження порогу: клампінг 0.35..0.65 для стабільного тест-порогу.
# ========================================================================

import os
import gc
import sys
import math
import time
import copy
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import tifffile
from PIL import Image
from scipy import ndimage
from scipy.ndimage import label as nd_label

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

# ========================================================================
# 1. КОНФІГУРАЦІЯ
# ========================================================================

VERSION = "22.1-GeneralizationFix"
SEED    = 2026

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything()

# Автоматичний пошук шляху до даних
CANDIDATE_ROOTS = [
    Path("/kaggle/input/competitions/vesuvius-challenge-ink-detection"),
    Path("/kaggle/input/vesuvius-challenge-ink-detection"),
    Path("C:/Users/sciga/vesuvius"),
    Path("./")
]
DATA_ROOT = next(
    (p for p in CANDIDATE_ROOTS if p.exists() and ((p / "train").exists() or (p / "1").exists())),
    CANDIDATE_ROOTS[0]
)

# Навчання на всіх 3 тренувальних фрагментах;
# Fragment 3 розбивається просторово: 80% висоти → train, 20% → val.
TRAIN_FRAGS      = ["1", "2", "3"]
VAL_FRAG         = "3"          # Фрагмент для валідації (тільки нижні 20%)
TRAIN_VAL_SPLIT  = 0.80         # 80% рядків фрагмента → train, 20% → val

PATCH = 256
DEPTH = 16  # 16 якірних шарів в активній зоні

EPOCHS       = 25
BATCH_SIZE   = 4
NUM_WORKERS  = 2 if os.name != "nt" else 0

LR           = 3.0e-4
WEIGHT_DECAY = 1.0e-4
GRAD_CLIP    = 1.0
EMA_DECAY    = 0.995

# Стратифікована вибірка патчів
TRAIN_PATCHES          = 4500
VAL_PATCHES            = 700
POS_PATCH_FRACTION     = 0.40
NEUTRAL_PATCH_FRACTION = 0.20
NEG_PATCH_FRACTION     = 0.40

POS_DENSITY        = 0.015
STRONG_POS_DENSITY = 0.05
MIN_VALID_DENSITY  = 0.35

# Кроки сканування
VAL_STRIDE  = 128
TEST_STRIDE = 64
THRESHOLDS  = np.arange(0.05, 0.951, 0.005)

# Клампінг порогу: не дозволяємо надто екстремальні значення
THRESHOLD_MIN = 0.35
THRESHOLD_MAX = 0.65

# ── Ваги лосів (v22.1: балансований Recall/Precision) ──────────────────
BCE_W       = 0.25
DICE_W      = 0.20   # було 0.15
TVERSKY_W   = 0.35   # було 0.30
FOCAL_W     = 0.10   # було 0.15
PRECISION_W = 0.05   # було 0.15 → менший тиск на Precision
AUX_W       = 0.10   # незмінно

# Tversky: α=β=0.5 → рівна вага FP та FN (було α=0.7, β=0.3)
TVERSKY_ALPHA = 0.5
TVERSKY_BETA  = 0.5
FOCAL_GAMMA   = 2.0

P_LOW  = 1.0
P_HIGH = 99.0

AUGMENT = True
MIN_COMPONENT_SIZE = 8

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP    = (DEVICE.type == "cuda")

if AMP:
    torch.backends.cudnn.benchmark = True

print("=" * 72)
print(f"SUBIT-Ink v{VERSION} | GENERALIZATION FIX ENGINE")
print(f"Device: {DEVICE} | AMP: {AMP} | Data Root: {DATA_ROOT}")
print(f"Patch: {PATCH}x{PATCH} | Depth Anchors: {DEPTH} | Epochs: {EPOCHS}")
print(f"Train frags: {TRAIN_FRAGS} | Val frag: {VAL_FRAG} (top {int((1-TRAIN_VAL_SPLIT)*100)}% rows)")
print(f"Loss: BCE={BCE_W} Dice={DICE_W} Tversky={TVERSKY_W}(a={TVERSKY_ALPHA}) Focal={FOCAL_W} PrecPres={PRECISION_W}")
print("=" * 72)

# ========================================================================
# 2. АКТИВНА ЗОНА Z-ШАРІВ + HANN ВІКНО
# ========================================================================

def active_depth_anchors(total_layers: int, num_anchors: int = DEPTH,
                         z_start_ratio=0.22, z_end_ratio=0.75):
    """Вибирає якірні шари з активної зони папірусу (14..48 для 65 шарів)."""
    if total_layers <= num_anchors:
        return list(range(total_layers))
    z_min = int(round(total_layers * z_start_ratio))
    z_max = int(round(total_layers * z_end_ratio))
    z_min = max(0, min(z_min, total_layers - 1))
    z_max = max(z_min + 1, min(z_max, total_layers - 1))
    anchors = sorted(set(np.linspace(z_min, z_max, num_anchors).round().astype(int).tolist()))
    return anchors

def make_hann_window(patch_size: int = PATCH) -> np.ndarray:
    """2D вікно Ганна для плавного зважування перекриттів патчів."""
    w1d = np.hanning(patch_size)
    w2d = np.outer(w1d, w1d).astype(np.float32)
    return np.clip(w2d, 0.05, 1.0)

HANN_WEIGHT = make_hann_window(PATCH)

# ========================================================================
# 3. ЗАВАНТАЖЕННЯ ФРАГМЕНТІВ
# ========================================================================

def find_fragment_dir(root, frag_id):
    root = Path(root)
    candidates = [
        root / "train" / str(frag_id),
        root / "test"  / str(frag_id),
        root / str(frag_id),
        root / "train" / f"fragment_{frag_id}",
        root / "test"  / f"fragment_{frag_id}",
    ]
    for p in candidates:
        if p.exists():
            return p
    matches = list(root.rglob(f"*{frag_id}*"))
    for p in matches:
        if p.is_dir() and ((p / "surface_volume").exists() or (p / "surface-volume").exists()):
            return p
    raise FileNotFoundError(f"Не знайдено фрагмент {frag_id} у {root}")

def find_surface_dir(fragment_dir):
    candidates = [
        fragment_dir / "surface_volume",
        fragment_dir / "surface-volume",
        fragment_dir / "surface_volumes",
    ]
    for p in candidates:
        if p.exists() and p.is_dir():
            return p
    raise FileNotFoundError(f"Не знайдено surface_volume у {fragment_dir}")

def find_label(fragment_dir, keyword):
    candidates = [
        fragment_dir / f"{keyword}.png",
        fragment_dir / f"{keyword}.tif",
        fragment_dir / f"{keyword}.tiff",
    ]
    for p in candidates:
        if p.exists():
            return p
    for p in fragment_dir.glob("*.png"):
        if keyword.lower() in p.name.lower():
            return p
    raise FileNotFoundError(f"Не знайдено {keyword} у {fragment_dir}")

def slice_key(p):
    nums = "".join(c if c.isdigit() else " " for c in p.stem).split()
    return int(nums[-1]) if nums else 0

class FragmentVolume:
    def __init__(self, frag_id):
        self.frag_id = str(frag_id)
        self.root = find_fragment_dir(DATA_ROOT, self.frag_id)
        self.surface_dir = find_surface_dir(self.root)

        self.slice_paths = sorted(
            [p for p in self.surface_dir.iterdir() if p.suffix.lower() in [".tif", ".tiff"]],
            key=slice_key
        )
        if len(self.slice_paths) == 0:
            raise RuntimeError(f"Не знайдено TIFF шарів у {self.surface_dir}")

        self.Z = len(self.slice_paths)
        self.anchors = active_depth_anchors(self.Z, DEPTH)
        self._slices = [None] * self.Z

        try:
            label_path = find_label(self.root, "inklabels")
            mask_path  = find_label(self.root, "mask")
            self.label = np.asarray(Image.open(label_path).convert("L"), dtype=np.uint8)
            self.mask  = np.asarray(Image.open(mask_path).convert("L"),  dtype=np.uint8)
            self.label = (self.label > 0).astype(np.uint8)
            self.mask  = (self.mask  > 0).astype(np.uint8)
            self.H, self.W = self.label.shape
        except Exception:
            arr = self._get_slice(0)
            self.H, self.W = arr.shape
            try:
                mask_path = find_label(self.root, "mask")
                self.mask = np.asarray(Image.open(mask_path).convert("L"), dtype=np.uint8)
                self.mask = (self.mask > 0).astype(np.uint8)
            except Exception:
                self.mask = np.ones((self.H, self.W), dtype=np.uint8)
            self.label = np.zeros((self.H, self.W), dtype=np.uint8)

        self.lo, self.hi = self._estimate_percentiles()
        ink_str = f" | Ink={self.label.mean()*100:.2f}%" if self.label.max() > 0 else ""
        print(f"  Фрагмент [{self.frag_id}]: Z={self.Z}, Anchors={self.anchors[:4]}..., "
              f"H={self.H}, W={self.W}{ink_str}")

    def _get_slice(self, z):
        if self._slices[z] is None:
            path = self.slice_paths[z]
            try:
                arr = tifffile.memmap(str(path))
            except Exception:
                arr = tifffile.imread(str(path))
            if arr.ndim > 2:
                arr = np.squeeze(arr)
            self._slices[z] = arr
        return self._slices[z]

    def _estimate_percentiles(self):
        zs = self.anchors[::4] if len(self.anchors) >= 4 else self.anchors
        vals = []
        for z in zs:
            arr = self._get_slice(int(z))
            h, w = arr.shape
            sh, sw = min(512, h), min(512, w)
            y0, x0 = max(0, (h - sh) // 2), max(0, (w - sw) // 2)
            crop = np.asarray(arr[y0:y0 + sh, x0:x0 + sw], dtype=np.float32).reshape(-1)
            if crop.size > 10000:
                crop = crop[np.random.choice(crop.size, 10000, replace=False)]
            vals.append(crop)
        vals = np.concatenate(vals)
        lo = np.percentile(vals, P_LOW)
        hi = np.percentile(vals, P_HIGH)
        if hi <= lo:
            hi = lo + 1.0
        return float(lo), float(hi)

    def read_patch(self, y, x, patch=PATCH, anchors=None):
        if anchors is None:
            anchors = self.anchors
        y = int(np.clip(y, 0, self.H - patch))
        x = int(np.clip(x, 0, self.W - patch))
        out = np.empty((len(anchors), patch, patch), dtype=np.float32)
        for i, z in enumerate(anchors):
            arr = self._get_slice(z)
            crop = np.asarray(arr[y:y + patch, x:x + patch], dtype=np.float32)
            crop = (crop - self.lo) / (self.hi - self.lo)
            crop = np.clip(crop, 0.0, 1.0)
            out[i] = crop
        mask  = self.mask[y:y + patch, x:x + patch]
        label = self.label[y:y + patch, x:x + patch]
        return out, mask.astype(np.float32), label.astype(np.float32)

# ========================================================================
# 4. СТРАТИФІКОВАНИЙ ДАТАСЕТ ПАТЧІВ (з підтримкою y-діапазону)
# ========================================================================

def build_patch_index(volumes, n_patches, seed=SEED, y_ranges=None):
    """
    y_ranges: список (y_min, y_max) на кожен том або None (повний діапазон).
    Це дозволяє використовувати лише тренувальну частину фрагментів.
    """
    rng = np.random.default_rng(seed)
    pos_target     = int(n_patches * POS_PATCH_FRACTION)
    neutral_target = int(n_patches * NEUTRAL_PATCH_FRACTION)
    neg_target     = n_patches - pos_target - neutral_target

    positive, neutral, negative = [], [], []

    for vid, vol in enumerate(volumes):
        # Визначаємо обмеження по y для цього тому
        if y_ranges is not None and vid < len(y_ranges):
            y_lo, y_hi = y_ranges[vid]
        else:
            y_lo, y_hi = 0, vol.H

        y_lo = max(0, y_lo)
        y_hi = min(vol.H, y_hi)

        step = PATCH // 2
        for y in range(y_lo, max(y_lo + 1, y_hi - PATCH + 1), step):
            for x in range(0, max(1, vol.W - PATCH + 1), step):
                yy = min(y, y_hi - PATCH)
                xx = min(x, vol.W - PATCH)
                if yy < 0:
                    continue
                m = vol.mask[yy:yy + PATCH, xx:xx + PATCH]
                if m.mean() < MIN_VALID_DENSITY:
                    continue
                lab = vol.label[yy:yy + PATCH, xx:xx + PATCH]
                d   = float(lab.mean())
                item = (vid, yy, xx, d)
                if d >= STRONG_POS_DENSITY:
                    positive.append(item)
                elif d >= POS_DENSITY:
                    neutral.append(item)
                else:
                    negative.append(item)

    rng.shuffle(positive)
    rng.shuffle(neutral)
    rng.shuffle(negative)

    result = []
    result.extend(positive[:min(pos_target, len(positive))])
    result.extend(neutral[:min(neutral_target, len(neutral))])
    result.extend(negative[:min(neg_target, len(negative))])

    pool = positive + neutral + negative
    rng.shuffle(pool)
    used = set((a, b, c) for a, b, c, _ in result)
    for item in pool:
        key = item[:3]
        if key in used:
            continue
        result.append(item)
        used.add(key)
        if len(result) >= n_patches:
            break

    rng.shuffle(result)
    return result[:n_patches]

class PatchDataset(Dataset):
    def __init__(self, volumes, index, augment=False):
        self.volumes = volumes
        self.index   = index
        self.augment = augment

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):
        vid, y, x, _ = self.index[idx]
        vol = self.volumes[vid]
        depth, valid, target = vol.read_patch(y, x, PATCH)

        if self.augment:
            k = np.random.randint(4)
            if k:
                depth  = np.rot90(depth,  k=k, axes=(1, 2)).copy()
                valid  = np.rot90(valid,  k=k).copy()
                target = np.rot90(target, k=k).copy()
            if np.random.rand() < 0.5:
                depth  = depth[:,  :, ::-1].copy()
                valid  = valid[:,     ::-1].copy()
                target = target[:,    ::-1].copy()
            if np.random.rand() < 0.5:
                depth  = depth[:,  ::-1, :].copy()
                valid  = valid[::-1,    :].copy()
                target = target[::-1,   :].copy()
            if np.random.rand() < 0.35:
                gain = np.random.uniform(0.92, 1.08)
                bias = np.random.uniform(-0.025, 0.025)
                depth = np.clip(depth * gain + bias, 0.0, 1.0)
            if np.random.rand() < 0.15:
                n_drop = np.random.randint(1, max(2, DEPTH // 8))
                ids = np.random.choice(DEPTH, n_drop, replace=False)
                depth[ids] = depth[max(0, DEPTH // 2)]

        depth  = torch.from_numpy(depth.copy()).float()
        valid  = torch.from_numpy(valid.copy()).float()
        target = torch.from_numpy(target.copy()).float()
        return depth, valid, target

# ========================================================================
# 5. АРХІТЕКТУРА: 3D DEPTH FUSION + SE-ATTENTION DECODER
# ========================================================================

class Conv3DBlock(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv3d(cin, cout, 3, padding=1, bias=False),
            nn.InstanceNorm3d(cout, affine=True),
            nn.GELU(),
            nn.Conv3d(cout, cout, 3, padding=1, bias=False),
            nn.InstanceNorm3d(cout, affine=True),
            nn.GELU()
        )
        self.skip = nn.Conv3d(cin, cout, 1) if cin != cout else nn.Identity()

    def forward(self, x):
        return self.block(x) + self.skip(x)

class DepthCollapse(nn.Module):
    def __init__(self, channels):
        super().__init__()
        hidden = max(8, channels // 4)
        self.attn = nn.Sequential(
            nn.Conv3d(channels, hidden, 1),
            nn.GELU(),
            nn.Conv3d(hidden, 1, 1)
        )

    def forward(self, x):
        scores  = self.attn(x)
        weights = torch.softmax(scores, dim=2)
        return (x * weights).sum(dim=2)

class SEBlock(nn.Module):
    """Squeeze-and-Excitation: вибіркове підсилення штрихів тексту."""
    def __init__(self, channels, reduction=8):
        super().__init__()
        self.fc = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, max(4, channels // reduction), 1),
            nn.GELU(),
            nn.Conv2d(max(4, channels // reduction), channels, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return x * self.fc(x)

class FusionBlock(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1, bias=False),
            nn.BatchNorm2d(cout),
            nn.GELU(),
            nn.Conv2d(cout, cout, 3, padding=1, bias=False),
            nn.BatchNorm2d(cout),
            nn.GELU(),
            SEBlock(cout)
        )

    def forward(self, x):
        return self.block(x)

class DecoderBlock(nn.Module):
    def __init__(self, cin, skip, cout):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(cin + skip, cout, 3, padding=1, bias=False),
            nn.BatchNorm2d(cout),
            nn.GELU(),
            nn.Conv2d(cout, cout, 3, padding=1, bias=False),
            nn.BatchNorm2d(cout),
            nn.GELU(),
            SEBlock(cout)
        )

    def forward(self, x, skip):
        x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        return self.conv(torch.cat([x, skip], dim=1))

class SUBITInkV22(nn.Module):
    def __init__(self):
        super().__init__()
        # 3D Depth Encoder
        self.enc1  = Conv3DBlock(1, 32)
        self.down1 = nn.Conv3d(32, 64, kernel_size=(1, 2, 2), stride=(1, 2, 2))
        self.enc2  = Conv3DBlock(64, 64)
        self.down2 = nn.Conv3d(64, 96, kernel_size=(1, 2, 2), stride=(1, 2, 2))
        self.enc3  = Conv3DBlock(96, 96)
        self.down3 = nn.Conv3d(96, 128, kernel_size=(1, 2, 2), stride=(1, 2, 2))
        self.enc4  = Conv3DBlock(128, 128)

        # Depth Collapse (attention-weighted sum over depth)
        self.collapse1 = DepthCollapse(32)
        self.collapse2 = DepthCollapse(64)
        self.collapse3 = DepthCollapse(96)
        self.collapse4 = DepthCollapse(128)

        # Spatial Projections з SE
        self.proj1 = FusionBlock(32,  48)
        self.proj2 = FusionBlock(64,  80)
        self.proj3 = FusionBlock(96,  112)
        self.proj4 = FusionBlock(128, 144)

        # U-Net Decoder з SE
        self.dec3 = DecoderBlock(144, 112, 112)
        self.dec2 = DecoderBlock(112,  80,  80)
        self.dec1 = DecoderBlock( 80,  48,  48)

        # Refinement head
        self.refine = nn.Sequential(
            nn.Conv2d(48, 48, 3, padding=1, bias=False),
            nn.BatchNorm2d(48),
            nn.GELU(),
            nn.Conv2d(48, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.GELU()
        )
        self.head = nn.Conv2d(32, 1, 1)

        # Deep Supervision
        self.aux3 = nn.Conv2d(112, 1, 1)
        self.aux2 = nn.Conv2d( 80, 1, 1)
        self.aux1 = nn.Conv2d( 32, 1, 1)

    def forward(self, x):
        x = x.unsqueeze(1)  # (B, 1, D, H, W)

        e1 = self.enc1(x)
        e2 = self.enc2(self.down1(e1))
        e3 = self.enc3(self.down2(e2))
        e4 = self.enc4(self.down3(e3))

        s1 = self.proj1(self.collapse1(e1))
        s2 = self.proj2(self.collapse2(e2))
        s3 = self.proj3(self.collapse3(e3))
        s4 = self.proj4(self.collapse4(e4))

        d3 = self.dec3(s4, s3)
        d2 = self.dec2(d3, s2)
        d1 = self.dec1(d2, s1)
        r  = self.refine(d1)
        logits = self.head(r)

        if self.training:
            return logits, self.aux3(d3), self.aux2(d2), self.aux1(r)
        return logits

# ========================================================================
# 6. КОМПЛЕКСНА ФУНКЦІЯ ВТРАТ ТА EMA
# ========================================================================

def masked_bce(logits, target, valid):
    loss = F.binary_cross_entropy_with_logits(logits, target, reduction="none")
    return (loss * valid).sum() / (valid.sum() + 1e-6)

def masked_dice(logits, target, valid):
    prob   = torch.sigmoid(logits) * valid
    target = target * valid
    inter  = (prob * target).sum(dim=(1, 2, 3))
    denom  = prob.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3)) + 1e-6
    return 1.0 - ((2.0 * inter + 1.0) / (denom + 1.0)).mean()

def masked_tversky(logits, target, valid,
                   alpha=TVERSKY_ALPHA, beta=TVERSKY_BETA):
    p      = torch.sigmoid(logits) * valid
    target = target * valid
    tp     = (p * target).sum(dim=(1, 2, 3))
    fp     = (p * (1.0 - target)).sum(dim=(1, 2, 3))
    fn     = ((1.0 - p) * target).sum(dim=(1, 2, 3))
    return 1.0 - ((tp + 1e-6) / (tp + alpha * fn + beta * fp + 1e-6)).mean()

def focal_loss_fn(logits, target, valid, gamma=FOCAL_GAMMA):
    prob = torch.sigmoid(logits)
    pt   = prob * target + (1.0 - prob) * (1.0 - target)
    bce  = F.binary_cross_entropy_with_logits(logits, target, reduction="none")
    return (((1.0 - pt).pow(gamma) * bce * valid).sum() / (valid.sum() + 1e-6))

def precision_pressure(logits, target, valid):
    """Слабкий тиск на Precision (ваговий коефіцієнт 0.05 у v22.1)."""
    prob = torch.sigmoid(logits)
    fp   = (prob * (1.0 - target) * valid).sum()
    tp   = (prob * target * valid).sum()
    return 1.0 - ((tp + 1.0) / (tp + fp + 1.0))

def total_loss(logits, target, valid, aux3=None, aux2=None, aux1=None):
    bce     = masked_bce(logits, target, valid)
    dice    = masked_dice(logits, target, valid)
    tversky = masked_tversky(logits, target, valid)
    focal   = focal_loss_fn(logits, target, valid)
    pp      = precision_pressure(logits, target, valid)

    loss = (BCE_W * bce + DICE_W * dice + TVERSKY_W * tversky +
            FOCAL_W * focal + PRECISION_W * pp)

    if aux3 is not None:
        sz = target.shape[-2:]
        a3 = F.interpolate(aux3, size=sz, mode="bilinear", align_corners=False)
        a2 = F.interpolate(aux2, size=sz, mode="bilinear", align_corners=False)
        a1 = F.interpolate(aux1, size=sz, mode="bilinear", align_corners=False)
        loss += AUX_W * (masked_bce(a3, target, valid) +
                         masked_bce(a2, target, valid) +
                         masked_bce(a1, target, valid))
    return loss

class ModelEMA:
    def __init__(self, model, decay=EMA_DECAY):
        self.decay = decay
        self.ema   = copy.deepcopy(model).eval()
        for p in self.ema.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        for (_, ep), (_, mp) in zip(self.ema.named_parameters(), model.named_parameters()):
            ep.mul_(self.decay).add_(mp, alpha=1.0 - self.decay)
        for (_, eb), (_, mb) in zip(self.ema.named_buffers(), model.named_buffers()):
            eb.copy_(mb)

# ========================================================================
# 7. ВАЛІДАЦІЯ З HANN BLENDING, TTA ТА ПРОСТОРОВОЮ МАСКОЮ
# ========================================================================

def binary_metrics(pred, target):
    pred   = pred.astype(bool)
    target = target.astype(bool)
    tp = np.logical_and(pred, target).sum()
    fp = np.logical_and(pred, ~target).sum()
    fn = np.logical_and(~pred, target).sum()
    precision = tp / (tp + fp + 1e-9)
    recall    = tp / (tp + fn + 1e-9)
    beta2 = 0.25
    f05 = (1.0 + beta2) * precision * recall / (beta2 * precision + recall + 1e-9)
    return float(f05), float(precision), float(recall), int(tp), int(fp), int(fn)

@torch.no_grad()
def infer_fragment_probability_map(eval_model, vol: FragmentVolume, stride=VAL_STRIDE):
    """Плавна 2D карта ймовірностей: 4x TTA + 2D вікно Ганна."""
    eval_model.eval()
    h, w = vol.H, vol.W
    prob_sum   = np.zeros((h, w), np.float32)
    weight_sum = np.zeros((h, w), np.float32)

    ys = list(range(0, max(1, h - PATCH + 1), stride))
    xs = list(range(0, max(1, w - PATCH + 1), stride))
    by, bx = max(0, h - PATCH), max(0, w - PATCH)
    if not ys or ys[-1] != by: ys.append(by)
    if not xs or xs[-1] != bx: xs.append(bx)

    total = len(ys) * len(xs)
    done  = 0

    for y in ys:
        for x in xs:
            depth, _, _ = vol.read_patch(y, x, PATCH)
            X = torch.from_numpy(depth).float().unsqueeze(0).to(DEVICE)

            probs = []
            for fh, fv in [(0, 0), (1, 0), (0, 1), (1, 1)]:
                Xt = X
                if fh: Xt = torch.flip(Xt, dims=[3])
                if fv: Xt = torch.flip(Xt, dims=[2])
                with torch.cuda.amp.autocast(enabled=AMP):
                    logit = eval_model(Xt)
                p = torch.sigmoid(logit)[0, 0]
                if fh: p = torch.flip(p, dims=[1])
                if fv: p = torch.flip(p, dims=[0])
                probs.append(p.cpu().numpy())

            p_avg   = np.mean(probs, axis=0)
            hh      = min(PATCH, h - y)
            ww      = min(PATCH, w - x)
            patch_w = HANN_WEIGHT[:hh, :ww]

            prob_sum  [y:y + hh, x:x + ww] += p_avg[:hh, :ww] * patch_w
            weight_sum[y:y + hh, x:x + ww] += patch_w
            done += 1
            if done % 150 == 0 or done == total:
                print(f"    inference: {done}/{total}", flush=True)

    prob = prob_sum / np.maximum(weight_sum, 1e-6)
    return prob

@torch.no_grad()
def validate_full(eval_model, vol: FragmentVolume, y_val_start: int = 0):
    """
    Валідація фрагмента.
    y_val_start > 0 → оцінюємо тільки рядки [y_val_start:] (просторовий holdout).
    """
    prob       = infer_fragment_probability_map(eval_model, vol, stride=VAL_STRIDE)
    full_label = vol.label
    full_mask  = vol.mask

    # Якщо задано просторовий holdout — маскуємо тренувальну область
    if y_val_start > 0:
        eval_mask = np.zeros_like(full_mask)
        eval_mask[y_val_start:] = full_mask[y_val_start:]
    else:
        eval_mask = full_mask

    best = None
    for t in THRESHOLDS:
        pred   = (prob >= t) & (eval_mask > 0)
        target = (full_label > 0) & (eval_mask > 0)
        f05, precision, recall, tp, fp, fn = binary_metrics(pred, target)
        if best is None or f05 > best["f05"]:
            best = {
                "threshold": float(t), "f05": f05,
                "precision": precision, "recall": recall,
                "tp": tp, "fp": fp, "fn": fn
            }
    return best, prob

# ========================================================================
# 8. ПОСТПРОЦЕСИНГ ТА RLE КОДУВАННЯ
# ========================================================================

def filter_small_connected_components(mask: np.ndarray,
                                      min_size: int = MIN_COMPONENT_SIZE) -> np.ndarray:
    if min_size <= 1:
        return mask
    labeled_arr, num_features = nd_label(mask)
    if num_features == 0:
        return mask
    component_sizes = np.bincount(labeled_arr.ravel())
    too_small      = component_sizes < min_size
    too_small_mask = too_small[labeled_arr]
    clean_mask = mask.copy()
    clean_mask[too_small_mask] = False
    return clean_mask

def postprocess_mask(prob: np.ndarray, threshold: float,
                     mask_region: np.ndarray) -> np.ndarray:
    mask = (prob >= float(threshold)) & (mask_region > 0)
    mask = ndimage.binary_closing(mask, iterations=1) & (mask_region > 0)
    if MIN_COMPONENT_SIZE > 1:
        mask = filter_small_connected_components(mask, min_size=MIN_COMPONENT_SIZE)
        mask = mask & (mask_region > 0)
    return mask

def rle_encode(mask: np.ndarray) -> str:
    pixels  = mask.astype(np.uint8).ravel()
    p       = np.concatenate([[0], pixels, [0]])
    changes = np.flatnonzero(p[1:] != p[:-1])
    starts  = changes[::2] + 1
    lengths = changes[1::2] - changes[::2]
    out = []
    for s, l in zip(starts, lengths):
        out.extend([int(s), int(l)])
    return " ".join(map(str, out))

# ========================================================================
# 9. ГОЛОВНИЙ ПАЙПЛАЙН: ПРОСТОРОВА CV + TOP-3 ENSEMBLE
# ========================================================================

def clamp_threshold(t: float) -> float:
    """Клампінг порогу в безпечний діапазон [THRESHOLD_MIN, THRESHOLD_MAX]."""
    return float(np.clip(t, THRESHOLD_MIN, THRESHOLD_MAX))

def main():
    # ── Завантаження томів ────────────────────────────────────────────────
    print("\nTRAIN ФРАГМЕНТИ:")
    all_volumes = [FragmentVolume(f) for f in TRAIN_FRAGS]

    # Знаходимо том валідаційного фрагмента
    val_frag_idx = TRAIN_FRAGS.index(VAL_FRAG)
    val_vol      = all_volumes[val_frag_idx]

    # Просторовий розподіл для фрагмента VAL_FRAG:
    # рядки 0..train_h   → тільки train-патчі
    # рядки train_h..H   → тільки val (holdout)
    train_h = int(val_vol.H * TRAIN_VAL_SPLIT)
    print(f"\n  Просторовий split для Fragment {VAL_FRAG}: "
          f"train_h={train_h} ({TRAIN_VAL_SPLIT*100:.0f}%), "
          f"val_h={val_vol.H - train_h} рядків")

    # y_ranges: для F1, F2 — повний діапазон; для F3 — тільки [0, train_h]
    train_y_ranges = []
    for i, fid in enumerate(TRAIN_FRAGS):
        if fid == VAL_FRAG:
            train_y_ranges.append((0, train_h))
        else:
            train_y_ranges.append((0, all_volumes[i].H))

    # ── Індекси патчів ────────────────────────────────────────────────────
    print("\nПобудова стратифікованих індексів патчів...")
    train_index = build_patch_index(
        all_volumes, TRAIN_PATCHES, seed=SEED, y_ranges=train_y_ranges
    )
    # Валідаційні патчі — тільки з holdout-зони F3
    val_index = build_patch_index(
        [val_vol], VAL_PATCHES, seed=SEED + 1,
        y_ranges=[(train_h, val_vol.H)]
    )
    print(f"Train патчів: {len(train_index):,} (F1+F2+F3-top-{int(TRAIN_VAL_SPLIT*100)}%)")
    print(f"Val патчів:   {len(val_index):,} (F3-bottom-{int((1-TRAIN_VAL_SPLIT)*100)}%)")

    train_ds = PatchDataset(all_volumes, train_index, augment=AUGMENT)
    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=AMP,
        persistent_workers=NUM_WORKERS > 0, drop_last=True
    )

    # ── Модель ────────────────────────────────────────────────────────────
    model = SUBITInkV22().to(DEVICE)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"\nПараметрів моделі (v22.1): {n_params:,}")

    ema       = ModelEMA(model)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=EPOCHS, eta_min=LR * 0.05
    )
    scaler = torch.cuda.amp.GradScaler(enabled=AMP)

    best_f05      = -1.0
    best_epoch    = -1
    best_threshold = 0.450
    best_state    = None
    top_checkpoints = []
    history       = []

    print("\n" + "=" * 72)
    print(f"НАВЧАННЯ SUBIT-Ink v{VERSION} ({EPOCHS} EPOCHS)")
    print("=" * 72)

    for epoch in range(1, EPOCHS + 1):
        t0 = time.time()
        model.train()
        total_loss_val = 0.0
        n_batches = 0

        for bi, batch in enumerate(train_loader, start=1):
            depth, valid, target = batch
            depth  = depth.to(DEVICE,  non_blocking=True)
            valid  = valid.to(DEVICE,  non_blocking=True)
            target = target.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=AMP):
                logits, aux3, aux2, aux1 = model(depth)
                loss = total_loss(
                    logits, target.unsqueeze(1), valid.unsqueeze(1),
                    aux3, aux2, aux1
                )

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
            ema.update(model)

            total_loss_val += float(loss.detach())
            n_batches += 1
            if bi == 1 or bi % 150 == 0 or bi == len(train_loader):
                print(f"    batch {bi}/{len(train_loader)} loss={total_loss_val / n_batches:.4f}")

        train_loss = total_loss_val / max(n_batches, 1)

        # ── Full overlap валідація на holdout-зоні Fragment 3 ─────────────
        print(f"\n  Валідація (Hann Window) на Fragment {VAL_FRAG} holdout (рядки {train_h}+)...")
        metrics, val_prob = validate_full(ema.ema, val_vol, y_val_start=train_h)
        scheduler.step()
        elapsed = (time.time() - t0) / 60.0

        print("\n" + "-" * 72)
        print(f"Epoch {epoch}/{EPOCHS} ({elapsed:.1f} хв)")
        print(f"Train loss={train_loss:.5f}")
        print(
            f"VAL HOLDOUT: F0.5={metrics['f05']:.4f} | "
            f"P={metrics['precision']:.4f} | "
            f"R={metrics['recall']:.4f} | "
            f"t={metrics['threshold']:.3f}"
        )
        print(f"TP={metrics['tp']} FP={metrics['fp']} FN={metrics['fn']}")
        print(f"LR={optimizer.param_groups[0]['lr']:.3e}")

        history.append({"epoch": epoch, "train_loss": train_loss, **metrics})

        # ── Збереження топ-3 чекпоїнтів (з кінця навчання) ──────────────
        curr_state = {k: v.detach().cpu().clone() for k, v in ema.ema.state_dict().items()}
        if epoch >= 10:
            top_checkpoints.append({
                "epoch":     epoch,
                "f05":       metrics["f05"],
                "threshold": metrics["threshold"],
                "state":     curr_state
            })
            top_checkpoints.sort(key=lambda c: c["f05"], reverse=True)
            if len(top_checkpoints) > 3:
                top_checkpoints = top_checkpoints[:3]
            ckpt_summary = [
                "Ep" + str(c["epoch"]) + ": " + f"{c['f05']:.4f}"
                for c in top_checkpoints
            ]
            print(f"    Топ чекпоїнти: {ckpt_summary}")

        if metrics["f05"] > best_f05:
            best_f05       = metrics["f05"]
            best_epoch     = epoch
            best_threshold = metrics["threshold"]
            best_state     = curr_state
            print(f"\n✓ НОВИЙ РЕКОРД F0.5: {best_f05:.4f} (t={best_threshold:.3f})")
            bad = 0
        else:
            bad = epoch - best_epoch
            print(f"\nБез покращення: {bad}/8")

        if bad >= 8 and epoch >= 18:
            print("\nEarly stopping triggered.")
            break

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # Якщо топ-3 не зібрались — беремо найкращий
    if not top_checkpoints and best_state is not None:
        top_checkpoints = [{
            "epoch": best_epoch, "f05": best_f05,
            "threshold": best_threshold, "state": best_state
        }]

    # ── Ансамбль ─────────────────────────────────────────────────────────
    print("\n" + "=" * 72)
    print(f"ФОРМУВАННЯ ТОП-{len(top_checkpoints)} АНСАМБЛЮ")
    print("=" * 72)

    ensemble_models = []
    for i, ckpt in enumerate(top_checkpoints, 1):
        m = SUBITInkV22().to(DEVICE).eval()
        m.load_state_dict(ckpt["state"])
        ensemble_models.append(m)
        print(f"  Модель {i}: Epoch {ckpt['epoch']} "
              f"(F0.5={ckpt['f05']:.4f}, t={ckpt['threshold']:.3f})")

    # Пошук оптимального порогу ансамблю на holdout-зоні F3
    print("\nПошук порогу ансамблю на val holdout...")
    ens_val_prob = np.zeros((val_vol.H, val_vol.W), np.float32)
    for m in ensemble_models:
        ens_val_prob += (
            infer_fragment_probability_map(m, val_vol, stride=VAL_STRIDE)
            / len(ensemble_models)
        )

    # Маска тільки holdout-зони
    holdout_mask = np.zeros_like(val_vol.mask)
    holdout_mask[train_h:] = val_vol.mask[train_h:]

    best_ens_t   = clamp_threshold(best_threshold)
    best_ens_f05 = -1.0
    for t in THRESHOLDS:
        pred   = (ens_val_prob >= t) & (holdout_mask > 0)
        target = (val_vol.label > 0) & (holdout_mask > 0)
        f05, p, r, _, _, _ = binary_metrics(pred, target)
        if f05 > best_ens_f05:
            best_ens_f05 = f05
            best_ens_t   = float(t)

    # Клампінг: не допускаємо екстремальних порогів
    raw_t      = best_ens_t
    best_ens_t = clamp_threshold(best_ens_t)
    if raw_t != best_ens_t:
        print(f"  ⚠ Поріг clamped: {raw_t:.3f} → {best_ens_t:.3f} "
              f"(допустимий діапазон {THRESHOLD_MIN:.2f}..{THRESHOLD_MAX:.2f})")

    print(f"\n Ансамбль Val F0.5={best_ens_f05:.4f} | Поріг={best_ens_t:.3f}")

    # ── Тестовий інференс ─────────────────────────────────────────────────
    print("\n" + "=" * 72)
    print("ФІНАЛЬНИЙ ТЕСТОВИЙ ІНФЕРЕНС (STRIDE=64, 4X TTA, TOP-3 ENSEMBLE)")
    print("=" * 72)

    test_root = Path(DATA_ROOT) / "test"
    test_ids  = (
        sorted([p.name for p in test_root.iterdir() if p.is_dir()])
        if test_root.exists() else ["a", "b"]
    )
    print(f"Тестових фрагментів: {len(test_ids)} ({test_ids})")

    rows = []
    for f in test_ids:
        print(f"\nОбробка тестового фрагмента: {f}...")
        frag = FragmentVolume(f)

        test_prob = np.zeros((frag.H, frag.W), np.float32)
        for i, m in enumerate(ensemble_models, 1):
            print(f"  Ансамбль прогін [{i}/{len(ensemble_models)}]...")
            test_prob += (
                infer_fragment_probability_map(m, frag, stride=TEST_STRIDE)
                / len(ensemble_models)
            )

        mask  = postprocess_mask(test_prob, best_ens_t, frag.mask)
        ratio = float(mask.mean())
        rle   = rle_encode(mask)
        print(f"  Поріг={best_ens_t:.3f} | Ink={ratio*100:.2f}% | RLE={len(rle):,} символів")
        rows.append({"Id": f, "Predicted": rle})

        del frag, test_prob, mask
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    submission = pd.DataFrame(rows, columns=["Id", "Predicted"])
    out_paths  = [Path("/kaggle/working/submission.csv"), Path("submission.csv")]
    for p in out_paths:
        try:
            submission.to_csv(p, index=False)
            print(f"Збережено {p}")
        except Exception:
            pass

    print("\n" + "=" * 72)
    print(f"submission.csv ГОТОВО! (v{VERSION})")
    print(f"Ensemble Val F0.5 : {best_ens_f05:.4f}")
    print(f"Ensemble Threshold: {best_ens_t:.3f}")
    print("=" * 72)
    print(submission.head())
    print("=" * 72)

if __name__ == "__main__":
    main()


========================================================================
SUBIT-Ink v22.1-GeneralizationFix | GENERALIZATION FIX ENGINE
Device: cuda | AMP: True | Data Root: /kaggle/input/competitions/vesuvius-challenge-ink-detection
Patch: 256x256 | Depth Anchors: 16 | Epochs: 25
Train frags: ['1', '2', '3'] | Val frag: 3 (bottom 19% rows)
Loss: BCE=0.25 Dice=0.2 Tversky=0.35(α=0.5) Focal=0.1 PrecPres=0.05
========================================================================

TRAIN ФРАГМЕНТИ:
  Fragment [1]: Z=65, Anchors=[14, 16, 19, 21]..., H=8181, W=6330 | Ink=10.31%
  Fragment [2]: Z=65, Anchors=[14, 16, 19, 21]..., H=14830, W=9506 | Ink=11.96%
  Fragment [3]: Z=65, Anchors=[14, 16, 19, 21]..., H=7606, W=5249 | Ink=7.95%

  Просторовий split для Fragment 3: train_h=6084 (80%), val_h=1522 rows

Побудова індексів патчів...
Train патчів: 4,500 (F1+F2+F3‑top‑80%)
Val   патчів: 168 (F3‑bottom‑19%)

Параметрів моделі (v22.1): 3,120,710

========================================================================
НАВЧАННЯ SUBIT‑Ink v22.1-GeneralizationFix (25 epochs)
========================================================================
    batch 1/1125 loss=0.7677
    batch 150/1125 loss=0.8350
    batch 300/1125 loss=0.8134
    batch 450/1125 loss=0.8028
    batch 600/1125 loss=0.7957
    batch 750/1125 loss=0.7901
    batch 900/1125 loss=0.7866
    batch 1050/1125 loss=0.7836
    batch 1125/1125 loss=0.7828

  Валідація (Hann) на Fragment 3 hold‑out (рядки 6084+)...
    inference: 150/2419
    inference: 300/2419
    inference: 450/2419
    inference: 600/2419
    inference: 750/2419
    inference: 900/2419
    inference: 1050/2419
    inference: 1200/2419
    inference: 1350/2419
    inference: 1500/2419
    inference: 1650/2419
    inference: 1800/2419
    inference: 1950/2419
    inference: 2100/2419
    inference: 2250/2419
    inference: 2400/2419
    inference: 2419/2419

------------------------------------------------------------------------
Epoch 1/25 (18.6 хв)
Train loss=0.78278
VAL HOLDOUT: F0.5=0.1942 | P=0.1761 | R=0.3289 | t=0.290
TP=98274 FP=459644 FN=200557
LR=2.989e-04

✓ НОВИЙ РЕКОРД F0.5: 0.1942 (t=0.290)
    batch 1/1125 loss=0.8967
    batch 150/1125 loss=0.7553
    batch 300/1125 loss=0.7545
    batch 450/1125 loss=0.7556
    batch 600/1125 loss=0.7558
    batch 750/1125 loss=0.7549
    batch 900/1125 loss=0.7549
    batch 1050/1125 loss=0.7544
    batch 1125/1125 loss=0.7544

  Валідація (Hann) на Fragment 3 hold‑out (рядки 6084+)...
    inference: 150/2419
    inference: 300/2419
    inference: 450/2419
    inference: 600/2419
    inference: 750/2419
    inference: 900/2419
    inference: 1050/2419
    inference: 1200/2419
    inference: 1350/2419
    inference: 1500/2419
    inference: 1650/2419
    inference: 1800/2419
    inference: 1950/2419
    inference: 2100/2419
    inference: 2250/2419
    inference: 2400/2419
    inference: 2419/2419

------------------------------------------------------------------------
Epoch 2/25 (17.8 хв)
Train loss=0.75439
VAL HOLDOUT: F0.5=0.2275 | P=0.2124 | R=0.3174 | t=0.405
TP=94863 FP=351767 FN=203968
LR=2.955e-04

✓ НОВИЙ РЕКОРД F0.5: 0.2275 (t=0.405)
    batch 1/1125 loss=0.7513
    batch 150/1125 loss=0.7580
    batch 300/1125 loss=0.7477
    batch 450/1125 loss=0.7458
    batch 600/1125 loss=0.7465
    batch 750/1125 loss=0.7453
    batch 900/1125 loss=0.7439
    batch 1050/1125 loss=0.7414
    batch 1125/1125 loss=0.7398

  Валідація (Hann) на Fragment 3 hold‑out (рядки 6084+)...
    inference: 150/2419
    inference: 300/2419
    inference: 450/2419
    inference: 600/2419
    inference: 750/2419
    inference: 900/2419
    inference: 1050/2419
    inference: 1200/2419
    inference: 1350/2419
    inference: 1500/2419
    inference: 1650/2419
    inference: 1800/2419
    inference: 1950/2419
    inference: 2100/2419
    inference: 2250/2419
    inference: 2400/2419
    inference: 2419/2419

------------------------------------------------------------------------
Epoch 3/25 (17.9 хв)
Train loss=0.73982
VAL HOLDOUT: F0.5=0.2776 | P=0.3237 | R=0.1768 | t=0.190
TP=52819 FP=110344 FN=246012
LR=2.900e-04

✓ НОВИЙ РЕКОРД F0.5: 0.2776 (t=0.190)
    batch 1/1125 loss=0.7365
    batch 150/1125 loss=0.7428
    batch 300/1125 loss=0.7348
    batch 450/1125 loss=0.7326
    batch 600/1125 loss=0.7313
    batch 750/1125 loss=0.7317
    batch 900/1125 loss=0.7328
    batch 1050/1125 loss=0.7324
    batch 1125/1125 loss=0.7326

  Валідація (Hann) на Fragment 3 hold‑out (рядки 6084+)...
    inference: 150/2419
    inference: 300/2419
    inference: 450/2419
    inference: 600/2419
    inference: 750/2419
    inference: 900/2419
    inference: 1050/2419
    inference: 1200/2419
    inference: 1350/2419
    inference: 1500/2419
    inference: 1650/2419
    inference: 1800/2419
    inference: 1950/2419
    inference: 2100/2419
    inference: 2250/2419
    inference: 2400/2419
    inference: 2419/2419

------------------------------------------------------------------------
Epoch 4/25 (17.8 хв)
Train loss=0.73264
VAL HOLDOUT: F0.5=0.4221 | P=0.5207 | R=0.2402 | t=0.560
TP=71793 FP=66086 FN=227038
LR=2.824e-04

✓ НОВИЙ РЕКОРД F0.5: 0.4221 (t=0.560)
    batch 1/1125 loss=0.7429
    batch 150/1125 loss=0.7243
    batch 300/1125 loss=0.7257
    batch 450/1125 loss=0.7237
    batch 600/1125 loss=0.7240
    batch 750/1125 loss=0.7228
    batch 900/1125 loss=0.7210
    batch 1050/1125 loss=0.7210
    batch 1125/1125 loss=0.7216

  Валідація (Hann) на Fragment 3 hold‑out (рядки 6084+)...
    inference: 150/2419
    inference: 300/2419
    inference: 450/2419
    inference: 600/2419
    inference: 750/2419
    inference: 900/2419
    inference: 1050/2419
    inference: 1200/2419
    inference: 1350/2419
    inference: 1500/2419
    inference: 1650/2419
    inference: 1800/2419
    inference: 1950/2419
    inference: 2100/2419
    inference: 2250/2419
    inference: 2400/2419
    inference: 2419/2419

------------------------------------------------------------------------
Epoch 5/25 (17.8 хв)
Train loss=0.72161
VAL HOLDOUT: F0.5=0.4817 | P=0.5997 | R=0.2696 | t=0.720
TP=80563 FP=53787 FN=218268
LR=2.728e-04

✓ НОВИЙ РЕКОРД F0.5: 0.4817 (t=0.720)
    batch 1/1125 loss=0.6584
    batch 150/1125 loss=0.7127
    batch 300/1125 loss=0.7140
    batch 450/1125 loss=0.7155
    batch 600/1125 loss=0.7173
    batch 750/1125 loss=0.7164
    batch 900/1125 loss=0.7136
    batch 1050/1125 loss=0.7129
    batch 1125/1125 loss=0.7132

  Валідація (Hann) на Fragment 3 hold‑out (рядки 6084+)...
    inference: 150/2419
    inference: 300/2419
    inference: 450/2419
    inference: 600/2419
    inference: 750/2419
    inference: 900/2419
    inference: 1050/2419
    inference: 1200/2419
    inference: 1350/2419
    inference: 1500/2419
    inference: 1650/2419
    inference: 1800/2419
    inference: 1950/2419
    inference: 2100/2419
    inference: 2250/2419
    inference: 2400/2419
    inference: 2419/2419

------------------------------------------------------------------------
Epoch 6/25 (17.8 хв)
Train loss=0.71316
VAL HOLDOUT: F0.5=0.4923 | P=0.6581 | R=0.2452 | t=0.820
TP=73274 FP=38063 FN=225557
LR=2.614e-04

✓ НОВИЙ РЕКОРД F0.5: 0.4923 (t=0.820)
    batch 1/1125 loss=0.6897
    batch 150/1125 loss=0.7057
    batch 300/1125 loss=0.7091
    batch 450/1125 loss=0.7091
    batch 600/1125 loss=0.7058
    batch 750/1125 loss=0.7039
    batch 900/1125 loss=0.7032
    batch 1050/1125 loss=0.7065
    batch 1125/1125 loss=0.7066

  Валідація (Hann) на Fragment 3 hold‑out (рядки 6084+)...
    inference: 150/2419
    inference: 300/2419
    inference: 450/2419
    inference: 600/2419
    inference: 750/2419
    inference: 900/2419
    inference: 1050/2419
    inference: 1200/2419
    inference: 1350/2419
    inference: 1500/2419
    inference: 1650/2419
    inference: 1800/2419
    inference: 1950/2419
    inference: 2100/2419
    inference: 2250/2419
    inference: 2400/2419
    inference: 2419/2419

------------------------------------------------------------------------
Epoch 7/25 (17.8 хв)
Train loss=0.70657
VAL HOLDOUT: F0.5=0.5519 | P=0.7311 | R=0.2787 | t=0.745
TP=83287 FP=30629 FN=215544
LR=2.483e-04

✓ НОВИЙ РЕКОРД F0.5: 0.5519 (t=0.745)
    batch 1/1125 loss=0.7231
    batch 150/1125 loss=0.6873
    batch 300/1125 loss=0.6905
    batch 450/1125 loss=0.6958
    batch 600/1125 loss=0.6981
    batch 750/1125 loss=0.6987
    batch 900/1125 loss=0.6990
    batch 1050/1125 loss=0.6988
    batch 1125/1125 loss=0.6991

  Валідація (Hann) на Fragment 3 hold‑out (рядки 6084+)...
    inference: 150/2419
    inference: 300/2419
    inference: 450/2419
    inference: 600/2419
    inference: 750/2419
    inference: 900/2419
    inference: 1050/2419
    inference: 1200/2419
    inference: 1350/2419
    inference: 1500/2419
    inference: 1650/2419
    inference: 1800/2419
    inference: 1950/2419
    inference: 2100/2419
    inference: 2250/2419
    inference: 2400/2419
    inference: 2419/2419

------------------------------------------------------------------------
Epoch 8/25 (17.8 хв)
Train loss=0.69906
VAL HOLDOUT: F0.5=0.6199 | P=0.7312 | R=0.3853 | t=0.470
TP=115125 FP=42325 FN=183706
LR=2.339e-04

✓ НОВИЙ РЕКОРД F0.5: 0.6199 (t=0.470)
    batch 1/1125 loss=0.7124
    batch 150/1125 loss=0.6873
    batch 300/1125 loss=0.6900
    batch 450/1125 loss=0.6962
    batch 600/1125 loss=0.6947
    batch 750/1125 loss=0.6941
    batch 900/1125 loss=0.6934
    batch 1050/1125 loss=0.6946
    batch 1125/1125 loss=0.6933

  Валідація (Hann) на Fragment 3 hold‑out (рядки 6084+)...
    inference: 150/2419
    inference: 300/2419
    inference: 450/2419
    inference: 600/2419
    inference: 750/2419
    inference: 900/2419
    inference: 1050/2419
    inference: 1200/2419
    inference: 1350/2419
    inference: 1500/2419
    inference: 1650/2419
    inference: 1800/2419
    inference: 1950/2419
    inference: 2100/2419
    inference: 2250/2419
    inference: 2400/2419
    inference: 2419/2419

------------------------------------------------------------------------
Epoch 9/25 (17.8 хв)
Train loss=0.69328
VAL HOLDOUT: F0.5=0.6515 | P=0.7526 | R=0.4238 | t=0.370
TP=126651 FP=41639 FN=172180
LR=2.182e-04

✓ НОВИЙ РЕКОРД F0.5: 0.6515 (t=0.370)
    batch 1/1125 loss=0.6976
    batch 150/1125 loss=0.6804
    batch 300/1125 loss=0.6839
    batch 450/1125 loss=0.6858
    batch 600/1125 loss=0.6824
    batch 750/1125 loss=0.6845
    batch 900/1125 loss=0.6864
    batch 1050/1125 loss=0.6852
    batch 1125/1125 loss=0.6844

  Валідація (Hann) на Fragment 3 hold‑out (рядки 6084+)...
    inference: 150/2419
    inference: 300/2419
    inference: 450/2419
    inference: 600/2419
    inference: 750/2419
    inference: 900/2419
    inference: 1050/2419
    inference: 1200/2419
    inference: 1350/2419
    inference: 1500/2419
    inference: 1650/2419
    inference: 1800/2419
    inference: 1950/2419
    inference: 2100/2419
    inference: 2250/2419
    inference: 2400/2419
    inference: 2419/2419

------------------------------------------------------------------------
Epoch 10/25 (17.8 хв)
Train loss=0.68441
VAL HOLDOUT: F0.5=0.6452 | P=0.7487 | R=0.4154 | t=0.385
TP=124148 FP=41665 FN=174683
LR=2.015e-04
    Топ‑чекпоінти: ['Ep10: 0.6452']

Без покращення: 1/8
    batch 1/1125 loss=0.5218
    batch 150/1125 loss=0.6936
    batch 300/1125 loss=0.6830
    batch 450/1125 loss=0.6769
    batch 600/1125 loss=0.6762
    batch 750/1125 loss=0.6764
    batch 900/1125 loss=0.6767
    batch 1050/1125 loss=0.6779
    batch 1125/1125 loss=0.6786

  Валідація (Hann) на Fragment 3 hold‑out (рядки 6084+)...
    inference: 150/2419
    inference: 300/2419
    inference: 450/2419
    inference: 600/2419
    inference: 750/2419
    inference: 900/2419
    inference: 1050/2419
    inference: 1200/2419
    inference: 1350/2419
    inference: 1500/2419
    inference: 1650/2419
    inference: 1800/2419
    inference: 1950/2419
    inference: 2100/2419
    inference: 2250/2419
    inference: 2400/2419
    inference: 2419/2419

------------------------------------------------------------------------
Epoch 11/25 (17.8 хв)
Train loss=0.67857
VAL HOLDOUT: F0.5=0.6583 | P=0.8459 | R=0.3488 | t=0.395
TP=104244 FP=18989 FN=194587
LR=1.842e-04
    Топ‑чекпоінти: ['Ep11: 0.6583', 'Ep10: 0.6452']

✓ НОВИЙ РЕКОРД F0.5: 0.6583 (t=0.395)
    batch 1/1125 loss=0.6792
    batch 150/1125 loss=0.6761
    batch 300/1125 loss=0.6757
    batch 450/1125 loss=0.6801
    batch 600/1125 loss=0.6791
    batch 750/1125 loss=0.6784
    batch 900/1125 loss=0.6746
    batch 1050/1125 loss=0.6737
    batch 1125/1125 loss=0.6745

  Валідація (Hann) на Fragment 3 hold‑out (рядки 6084+)...
    inference: 150/2419
    inference: 300/2419
    inference: 450/2419
    inference: 600/2419
    inference: 750/2419
    inference: 900/2419
    inference: 1050/2419
    inference: 1200/2419
    inference: 1350/2419
    inference: 1500/2419
    inference: 1650/2419
    inference: 1800/2419
    inference: 1950/2419
    inference: 2100/2419
    inference: 2250/2419
    inference: 2400/2419
    inference: 2419/2419

------------------------------------------------------------------------
Epoch 12/25 (17.9 хв)
Train loss=0.67453
VAL HOLDOUT: F0.5=0.6650 | P=0.7738 | R=0.4256 | t=0.335
TP=127177 FP=37168 FN=171654
LR=1.664e-04
    Топ‑чекпоінти: ['Ep12: 0.6650', 'Ep11: 0.6583', 'Ep10: 0.6452']

✓ НОВИЙ РЕКОРД F0.5: 0.6650 (t=0.335)
    batch 1/1125 loss=0.5729
    batch 150/1125 loss=0.6699
    batch 300/1125 loss=0.6692
    batch 450/1125 loss=0.6697
    batch 600/1125 loss=0.6714
    batch 750/1125 loss=0.6717
    batch 900/1125 loss=0.6698
    batch 1050/1125 loss=0.6671
    batch 1125/1125 loss=0.6678

  Валідація (Hann) на Fragment 3 hold‑out (рядки 6084+)...
    inference: 150/2419
    inference: 300/2419
    inference: 450/2419
    inference: 600/2419
    inference: 750/2419
    inference: 900/2419
    inference: 1050/2419
    inference: 1200/2419
    inference: 1350/2419
    inference: 1500/2419
    inference: 1650/2419
    inference: 1800/2419
    inference: 1950/2419
    inference: 2100/2419
    inference: 2250/2419
    inference: 2400/2419
    inference: 2419/2419

------------------------------------------------------------------------
Epoch 13/25 (17.9 хв)
Train loss=0.66783
VAL HOLDOUT: F0.5=0.6706 | P=0.7985 | R=0.4088 | t=0.630
TP=122157 FP=30833 FN=176674
LR=1.486e-04
    Топ‑чекпоінти: ['Ep13: 0.6706', 'Ep12: 0.6650', 'Ep11: 0.6583']

✓ НОВИЙ РЕКОРД F0.5: 0.6706 (t=0.630)
    batch 1/1125 loss=0.8225
    batch 150/1125 loss=0.6620
    batch 300/1125 loss=0.6567
    batch 450/1125 loss=0.6581
    batch 600/1125 loss=0.6615
    batch 750/1125 loss=0.6610
    batch 900/1125 loss=0.6614
    batch 1050/1125 loss=0.6601
    batch 1125/1125 loss=0.6601

  Валідація (Hann) на Fragment 3 hold‑out (рядки 6084+)...
    inference: 150/2419
    inference: 300/2419
    inference: 450/2419
    inference: 600/2419
    inference: 750/2419
    inference: 900/2419
    inference: 1050/2419
    inference: 1200/2419
    inference: 1350/2419
    inference: 1500/2419
    inference: 1650/2419
    inference: 1800/2419
    inference: 1950/2419
    inference: 2100/2419
    inference: 2250/2419
    inference: 2400/2419
    inference: 2419/2419

------------------------------------------------------------------------
Epoch 14/25 (17.9 хв)
Train loss=0.66012
VAL HOLDOUT: F0.5=0.6614 | P=0.8497 | R=0.3507 | t=0.185
TP=104793 FP=18537 FN=194038
LR=1.308e-04
    Топ‑чекпоінти: ['Ep13: 0.6706', 'Ep12: 0.6650', 'Ep14: 0.6614']

Без покращення: 1/8
    batch 1/1125 loss=0.6746
    batch 150/1125 loss=0.6536
    batch 300/1125 loss=0.6555
    batch 450/1125 loss=0.6575
    batch 600/1125 loss=0.6559
    batch 750/1125 loss=0.6536
    batch 900/1125 loss=0.6530
    batch 1050/1125 loss=0.6545
    batch 1125/1125 loss=0.6542

  Валідація (Hann) на Fragment 3 hold‑out (рядки 6084+)...
    inference: 150/2419
    inference: 300/2419
    inference: 450/2419
    inference: 600/2419
    inference: 750/2419
    inference: 900/2419
    inference: 1050/2419
    inference: 1200/2419
    inference: 1350/2419
    inference: 1500/2419
    inference: 1650/2419
    inference: 1800/2419
    inference: 1950/2419
    inference: 2100/2419
    inference: 2250/2419
    inference: 2400/2419
    inference: 2419/2419

------------------------------------------------------------------------
Epoch 15/25 (17.9 хв)
Train loss=0.65422
VAL HOLDOUT: F0.5=0.6851 | P=0.8142 | R=0.4191 | t=0.585
TP=125247 FP=28578 FN=173584
LR=1.135e-04
    Топ‑чекпоінти: ['Ep15: 0.6851', 'Ep13: 0.6706', 'Ep12: 0.6650']

✓ НОВИЙ РЕКОРД F0.5: 0.6851 (t=0.585)
    batch 1/1125 loss=0.5368
    batch 150/1125 loss=0.6627
    batch 300/1125 loss=0.6566
    batch 450/1125 loss=0.6563
    batch 600/1125 loss=0.6540
    batch 750/1125 loss=0.6507
    batch 900/1125 loss=0.6514
    batch 1050/1125 loss=0.6515
    batch 1125/1125 loss=0.6523

  Валідація (Hann) на Fragment 3 hold‑out (рядки 6084+)...
    inference: 150/2419
    inference: 300/2419
    inference: 450/2419
    inference: 600/2419
    inference: 750/2419
    inference: 900/2419
    inference: 1050/2419
    inference: 1200/2419
    inference: 1350/2419
    inference: 1500/2419
    inference: 1650/2419
    inference: 1800/2419
    inference: 1950/2419
    inference: 2100/2419
    inference: 2250/2419
    inference: 2400/2419
    inference: 2419/2419

------------------------------------------------------------------------
Epoch 16/25 (17.9 хв)
Train loss=0.65227
VAL HOLDOUT: F0.5=0.6917 | P=0.8028 | R=0.4452 | t=0.420
TP=133027 FP=32673 FN=165804
LR=9.683e-05
    Топ‑чекпоінти: ['Ep16: 0.6917', 'Ep15: 0.6851', 'Ep13: 0.6706']

✓ НОВИЙ РЕКОРД F0.5: 0.6917 (t=0.420)
    batch 1/1125 loss=0.5857
    batch 150/1125 loss=0.6351
    batch 300/1125 loss=0.6358
    batch 450/1125 loss=0.6435
    batch 600/1125 loss=0.6442
    batch 750/1125 loss=0.6470
    batch 900/1125 loss=0.6473
    batch 1050/1125 loss=0.6477
    batch 1125/1125 loss=0.6494

  Валідація (Hann) на Fragment 3 hold‑out (рядки 6084+)...
    inference: 150/2419
    inference: 300/2419
    inference: 450/2419
    inference: 600/2419
    inference: 750/2419
    inference: 900/2419
    inference: 1050/2419
    inference: 1200/2419
    inference: 1350/2419
    inference: 1500/2419
    inference: 1650/2419
    inference: 1800/2419
    inference: 1950/2419
    inference: 2100/2419
    inference: 2250/2419
    inference: 2400/2419
    inference: 2419/2419

------------------------------------------------------------------------
Epoch 17/25 (17.9 хв)
Train loss=0.64943
VAL HOLDOUT: F0.5=0.6870 | P=0.7867 | R=0.4559 | t=0.530
TP=136247 FP=36934 FN=162584
LR=8.114e-05
    Топ‑чекпоінти: ['Ep16: 0.6917', 'Ep17: 0.6870', 'Ep15: 0.6851']

Без покращення: 1/8
    batch 1/1125 loss=0.6520
    batch 150/1125 loss=0.6430
    batch 300/1125 loss=0.6497
    batch 450/1125 loss=0.6488
    batch 600/1125 loss=0.6434
    batch 750/1125 loss=0.6419
    batch 900/1125 loss=0.6395
    batch 1050/1125 loss=0.6396
    batch 1125/1125 loss=0.6397

  Валідація (Hann) на Fragment 3 hold‑out (рядки 6084+)...
    inference: 150/2419
    inference: 300/2419
    inference: 450/2419
    inference: 600/2419
    inference: 750/2419
    inference: 900/2419
    inference: 1050/2419
    inference: 1200/2419
    inference: 1350/2419
    inference: 1500/2419
    inference: 1650/2419
    inference: 1800/2419
    inference: 1950/2419
    inference: 2100/2419
    inference: 2250/2419
    inference: 2400/2419
    inference: 2419/2419

------------------------------------------------------------------------
Epoch 18/25 (17.8 хв)
Train loss=0.63967
VAL HOLDOUT: F0.5=0.7038 | P=0.8105 | R=0.4610 | t=0.645
TP=137770 FP=32208 FN=161061
LR=6.667e-05
    Топ‑чекпоінти: ['Ep18: 0.7038', 'Ep16: 0.6917', 'Ep17: 0.6870']

✓ НОВИЙ РЕКОРД F0.5: 0.7038 (t=0.645)
    batch 1/1125 loss=0.9337
    batch 150/1125 loss=0.6379
    batch 300/1125 loss=0.6362
    batch 450/1125 loss=0.6361
    batch 600/1125 loss=0.6361
    batch 750/1125 loss=0.6341
    batch 900/1125 loss=0.6361
    batch 1050/1125 loss=0.6359
    batch 1125/1125 loss=0.6351

  Валідація (Hann) на Fragment 3 hold‑out (рядки 6084+)...
    inference: 150/2419
    inference: 300/2419
    inference: 450/2419
    inference: 600/2419
    inference: 750/2419
    inference: 900/2419
    inference: 1050/2419
    inference: 1200/2419
    inference: 1350/2419
    inference: 1500/2419
    inference: 1650/2419
    inference: 1800/2419
    inference: 1950/2419
    inference: 2100/2419
    inference: 2250/2419
    inference: 2400/2419
    inference: 2419/2419

------------------------------------------------------------------------
Epoch 19/25 (17.9 хв)
Train loss=0.63507
VAL HOLDOUT: F0.5=0.6875 | P=0.8484 | R=0.3910 | t=0.400
TP=116837 FP=20884 FN=181994
LR=5.362e-05
    Топ‑чекпоінти: ['Ep18: 0.7038', 'Ep16: 0.6917', 'Ep19: 0.6875']

Без покращення: 1/8
    batch 1/1125 loss=0.5355
    batch 150/1125 loss=0.6317
    batch 300/1125 loss=0.6274
    batch 450/1125 loss=0.6304
    batch 600/1125 loss=0.6340
    batch 750/1125 loss=0.6327
    batch 900/1125 loss=0.6330
    batch 1050/1125 loss=0.6321
    batch 1125/1125 loss=0.6299

  Валідація (Hann) на Fragment 3 hold‑out (рядки 6084+)...
    inference: 150/2419
    inference: 300/2419
    inference: 450/2419
    inference: 600/2419
    inference: 750/2419
    inference: 900/2419
    inference: 1050/2419
    inference: 1200/2419
    inference: 1350/2419
    inference: 1500/2419
    inference: 1650/2419
    inference: 1800/2419
    inference: 1950/2419
    inference: 2100/2419
    inference: 2250/2419
    inference: 2400/2419
    inference: 2419/2419

------------------------------------------------------------------------
Epoch 20/25 (17.9 хв)
Train loss=0.62992
VAL HOLDOUT: F0.5=0.6940 | P=0.8134 | R=0.4372 | t=0.675
TP=130641 FP=29967 FN=168190
LR=4.222e-05
    Топ‑чекпоінти: ['Ep18: 0.7038', 'Ep20: 0.6940', 'Ep16: 0.6917']

Без покращення: 2/8
    batch 1/1125 loss=0.4607
    batch 150/1125 loss=0.6263
    batch 300/1125 loss=0.6265
    batch 450/1125 loss=0.6222
    batch 600/1125 loss=0.6227
    batch 750/1125 loss=0.6228
    batch 900/1125 loss=0.6221
    batch 1050/1125 loss=0.6222
    batch 1125/1125 loss=0.6236

  Валідація (Hann) на Fragment 3 hold‑out (рядки 6084+)...
    inference: 150/2419
    inference: 300/2419
    inference: 450/2419
    inference: 600/2419
    inference: 750/2419
    inference: 900/2419
    inference: 1050/2419
    inference: 1200/2419
    inference: 1350/2419
    inference: 1500/2419
    inference: 1650/2419
    inference: 1800/2419
    inference: 1950/2419
    inference: 2100/2419
    inference: 2250/2419
    inference: 2400/2419
    inference: 2419/2419

------------------------------------------------------------------------
Epoch 21/25 (17.9 хв)
Train loss=0.62355
VAL HOLDOUT: F0.5=0.6931 | P=0.8251 | R=0.4227 | t=0.685
TP=126302 FP=26773 FN=172529
LR=3.263e-05
    Топ‑чекпоінти: ['Ep18: 0.7038', 'Ep20: 0.6940', 'Ep21: 0.6931']

Без покращення: 3/8
    batch 1/1125 loss=0.8592
    batch 150/1125 loss=0.6223
    batch 300/1125 loss=0.6194
    batch 450/1125 loss=0.6214
    batch 600/1125 loss=0.6241
    batch 750/1125 loss=0.6244
    batch 900/1125 loss=0.6263
    batch 1050/1125 loss=0.6234
    batch 1125/1125 loss=0.6250

  Валідація (Hann) на Fragment 3 hold‑out (рядки 6084+)...
    inference: 150/2419
    inference: 300/2419
    inference: 450/2419
    inference: 600/2419
    inference: 750/2419
    inference: 900/2419
    inference: 1050/2419
    inference: 1200/2419
    inference: 1350/2419
    inference: 1500/2419
    inference: 1650/2419
    inference: 1800/2419
    inference: 1950/2419
    inference: 2100/2419
    inference: 2250/2419
    inference: 2400/2419
    inference: 2419/2419

------------------------------------------------------------------------
Epoch 22/25 (17.9 хв)
Train loss=0.62497
VAL HOLDOUT: F0.5=0.6989 | P=0.8403 | R=0.4177 | t=0.490
TP=124817 FP=23716 FN=174014
LR=2.501e-05
    Топ‑чекпоінти: ['Ep18: 0.7038', 'Ep22: 0.6989', 'Ep20: 0.6940']

Без покращення: 4/8
    batch 1/1125 loss=0.4962
    batch 150/1125 loss=0.6325
    batch 300/1125 loss=0.6217
    batch 450/1125 loss=0.6244
    batch 600/1125 loss=0.6212
    batch 750/1125 loss=0.6222
    batch 900/1125 loss=0.6220
    batch 1050/1125 loss=0.6217
    batch 1125/1125 loss=0.6210

  Валідація (Hann) на Fragment 3 hold‑out (рядки 6084+)...
    inference: 150/2419
    inference: 300/2419
    inference: 450/2419
    inference: 600/2419
    inference: 750/2419
    inference: 900/2419
    inference: 1050/2419
    inference: 1200/2419
    inference: 1350/2419
    inference: 1500/2419
    inference: 1650/2419
    inference: 1800/2419
    inference: 1950/2419
    inference: 2100/2419
    inference: 2250/2419
    inference: 2400/2419
    inference: 2419/2419

------------------------------------------------------------------------
Epoch 23/25 (17.8 хв)
Train loss=0.62103
VAL HOLDOUT: F0.5=0.6837 | P=0.8312 | R=0.3998 | t=0.450
TP=119474 FP=24259 FN=179357
LR=1.948e-05
    Топ‑чекпоінти: ['Ep18: 0.7038', 'Ep22: 0.6989', 'Ep20: 0.6940']

Без покращення: 5/8
    batch 1/1125 loss=0.7952
    batch 150/1125 loss=0.6222
    batch 300/1125 loss=0.6143
    batch 450/1125 loss=0.6187
    batch 600/1125 loss=0.6213
    batch 750/1125 loss=0.6179
    batch 900/1125 loss=0.6173
    batch 1050/1125 loss=0.6165
    batch 1125/1125 loss=0.6166

  Валідація (Hann) на Fragment 3 hold‑out (рядки 6084+)...
    inference: 150/2419
    inference: 300/2419
    inference: 450/2419
    inference: 600/2419
    inference: 750/2419
    inference: 900/2419
    inference: 1050/2419
    inference: 1200/2419
    inference: 1350/2419
    inference: 1500/2419
    inference: 1650/2419
    inference: 1800/2419
    inference: 1950/2419
    inference: 2100/2419
    inference: 2250/2419
    inference: 2400/2419
    inference: 2419/2419

------------------------------------------------------------------------
Epoch 24/25 (17.8 хв)
Train loss=0.61661
VAL HOLDOUT: F0.5=0.6913 | P=0.8389 | R=0.4058 | t=0.490
TP=121270 FP=23294 FN=177561
LR=1.612e-05
    Топ‑чекпоінти: ['Ep18: 0.7038', 'Ep22: 0.6989', 'Ep20: 0.6940']

Без покращення: 6/8
    batch 1/1125 loss=0.6560
    batch 150/1125 loss=0.6088
    batch 300/1125 loss=0.6189
    batch 450/1125 loss=0.6188
    batch 600/1125 loss=0.6168
    batch 750/1125 loss=0.6157
    batch 900/1125 loss=0.6173
    batch 1050/1125 loss=0.6164
    batch 1125/1125 loss=0.6185

  Валідація (Hann) на Fragment 3 hold‑out (рядки 6084+)...
    inference: 150/2419
    inference: 300/2419
    inference: 450/2419
    inference: 600/2419
    inference: 750/2419
    inference: 900/2419
    inference: 1050/2419
    inference: 1200/2419
    inference: 1350/2419
    inference: 1500/2419
    inference: 1650/2419
    inference: 1800/2419
    inference: 1950/2419
    inference: 2100/2419
    inference: 2250/2419
    inference: 2400/2419
    inference: 2419/2419

------------------------------------------------------------------------
Epoch 25/25 (17.9 хв)
Train loss=0.61854
VAL HOLDOUT: F0.5=0.6941 | P=0.8465 | R=0.4034 | t=0.595
TP=120556 FP=21854 FN=178275
LR=1.500e-05
    Топ‑чекпоінти: ['Ep18: 0.7038', 'Ep22: 0.6989', 'Ep25: 0.6941']

Без покращення: 7/8

========================================================================
ФОРМУВАННЯ ТОП-3 АНСАМБЛЮ
========================================================================
  Модель 1: Epoch 18 (F0.5=0.7038, t=0.645)
  Модель 2: Epoch 22 (F0.5=0.6989, t=0.490)
  Модель 3: Epoch 25 (F0.5=0.6941, t=0.595)

Пошук порогу ансамблю на hold‑out...
    inference: 150/2419
    inference: 300/2419
    inference: 450/2419
    inference: 600/2419
    inference: 750/2419
    inference: 900/2419
    inference: 1050/2419
    inference: 1200/2419
    inference: 1350/2419
    inference: 1500/2419
    inference: 1650/2419
    inference: 1800/2419
    inference: 1950/2419
    inference: 2100/2419
    inference: 2250/2419
    inference: 2400/2419
    inference: 2419/2419
    inference: 150/2419
    inference: 300/2419
    inference: 450/2419
    inference: 600/2419
    inference: 750/2419
    inference: 900/2419
    inference: 1050/2419
    inference: 1200/2419
    inference: 1350/2419
    inference: 1500/2419
    inference: 1650/2419
    inference: 1800/2419
    inference: 1950/2419
    inference: 2100/2419
    inference: 2250/2419
    inference: 2400/2419
    inference: 2419/2419
    inference: 150/2419
    inference: 300/2419
    inference: 450/2419
    inference: 600/2419
    inference: 750/2419
    inference: 900/2419
    inference: 1050/2419
    inference: 1200/2419
    inference: 1350/2419
    inference: 1500/2419
    inference: 1650/2419
    inference: 1800/2419
    inference: 1950/2419
    inference: 2100/2419
    inference: 2250/2419
    inference: 2400/2419
    inference: 2419/2419

Ансамбль Val F0.5=0.6967 | Поріг=0.600

========================================================================
ФІНАЛЬНИЙ ТЕСТОВИЙ ІНФЕРЕНС (STRIDE=64, 4×TTA, TOP‑3 ENSEMBLE)
========================================================================
Тестових фрагментів: 2 (['a', 'b'])

Обробка тестового фрагмента: a...
  Fragment [a]: Z=65, Anchors=[14, 16, 19, 21]..., H=2727, W=6330
  Ансамбль прогін [1/3]...
    inference: 150/3840
    inference: 300/3840
    inference: 450/3840
    inference: 600/3840
    inference: 750/3840
    inference: 900/3840
    inference: 1050/3840
    inference: 1200/3840
    inference: 1350/3840
    inference: 1500/3840
    inference: 1650/3840
    inference: 1800/3840
    inference: 1950/3840
    inference: 2100/3840
    inference: 2250/3840
    inference: 2400/3840
    inference: 2550/3840
    inference: 2700/3840
    inference: 2850/3840
    inference: 3000/3840
    inference: 3150/3840
    inference: 3300/3840
    inference: 3450/3840
    inference: 3600/3840
    inference: 3750/3840
    inference: 3840/3840
  Ансамбль прогін [2/3]...
    inference: 150/3840
    inference: 300/3840
    inference: 450/3840
    inference: 600/3840
    inference: 750/3840
    inference: 900/3840
    inference: 1050/3840
    inference: 1200/3840
    inference: 1350/3840
    inference: 1500/3840
    inference: 1650/3840
    inference: 1800/3840
    inference: 1950/3840
    inference: 2100/3840
    inference: 2250/3840
    inference: 2400/3840
    inference: 2550/3840
    inference: 2700/3840
    inference: 2850/3840
    inference: 3000/3840
    inference: 3150/3840
    inference: 3300/3840
    inference: 3450/3840
    inference: 3600/3840
    inference: 3750/3840
    inference: 3840/3840
  Ансамбль прогін [3/3]...
    inference: 150/3840
    inference: 300/3840
    inference: 450/3840
    inference: 600/3840
    inference: 750/3840
    inference: 900/3840
    inference: 1050/3840
    inference: 1200/3840
    inference: 1350/3840
    inference: 1500/3840
    inference: 1650/3840
    inference: 1800/3840
    inference: 1950/3840
    inference: 2100/3840
    inference: 2250/3840
    inference: 2400/3840
    inference: 2550/3840
    inference: 2700/3840
    inference: 2850/3840
    inference: 3000/3840
    inference: 3150/3840
    inference: 3300/3840
    inference: 3450/3840
    inference: 3600/3840
    inference: 3750/3840
    inference: 3840/3840
  Поріг=0.600 | Ink=5.55% | RLE=119,040 символів

Обробка тестового фрагмента: b...
  Fragment [b]: Z=65, Anchors=[14, 16, 19, 21]..., H=5454, W=6330
  Ансамбль прогін [1/3]...
    inference: 150/7968
    inference: 300/7968
    inference: 450/7968
    inference: 600/7968
    inference: 750/7968
    inference: 900/7968
    inference: 1050/7968
    inference: 1200/7968
    inference: 1350/7968
    inference: 1500/7968
    inference: 1650/7968
    inference: 1800/7968
    inference: 1950/7968
    inference: 2100/7968
    inference: 2250/7968
    inference: 2400/7968
    inference: 2550/7968
    inference: 2700/7968
    inference: 2850/7968
    inference: 3000/7968
    inference: 3150/7968
    inference: 3300/7968
    inference: 3450/7968
    inference: 3600/7968
    inference: 3750/7968
    inference: 3900/7968
    inference: 4050/7968
    inference: 4200/7968
    inference: 4350/7968
    inference: 4500/7968
    inference: 4650/7968
    inference: 4800/7968
    inference: 4950/7968
    inference: 5100/7968
    inference: 5250/7968
    inference: 5400/7968
    inference: 5550/7968
    inference: 5700/7968
    inference: 5850/7968
    inference: 6000/7968
    inference: 6150/7968
    inference: 6300/7968
    inference: 6450/7968
    inference: 6600/7968
    inference: 6750/7968
    inference: 6900/7968
    inference: 7050/7968
    inference: 7200/7968
    inference: 7350/7968
    inference: 7500/7968
    inference: 7650/7968
    inference: 7800/7968
    inference: 7950/7968
    inference: 7968/7968
  Ансамбль прогін [2/3]...
    inference: 150/7968
    inference: 300/7968
    inference: 450/7968
    inference: 600/7968
    inference: 750/7968
    inference: 900/7968
    inference: 1050/7968
    inference: 1200/7968
    inference: 1350/7968
    inference: 1500/7968
    inference: 1650/7968
    inference: 1800/7968
    inference: 1950/7968
    inference: 2100/7968
    inference: 2250/7968
    inference: 2400/7968
    inference: 2550/7968
    inference: 2700/7968
    inference: 2850/7968
    inference: 3000/7968
    inference: 3150/7968
    inference: 3300/7968
    inference: 3450/7968
    inference: 3600/7968
    inference: 3750/7968
    inference: 3900/7968
    inference: 4050/7968
    inference: 4200/7968
    inference: 4350/7968
    inference: 4500/7968
    inference: 4650/7968
    inference: 4800/7968
    inference: 4950/7968
    inference: 5100/7968
    inference: 5250/7968
    inference: 5400/7968
    inference: 5550/7968
    inference: 5700/7968
    inference: 5850/7968
    inference: 6000/7968
    inference: 6150/7968
    inference: 6300/7968
    inference: 6450/7968
    inference: 6600/7968
    inference: 6750/7968
    inference: 6900/7968
    inference: 7050/7968
    inference: 7200/7968
    inference: 7350/7968
    inference: 7500/7968
    inference: 7650/7968
    inference: 7800/7968
    inference: 7950/7968
    inference: 7968/7968
  Ансамбль прогін [3/3]...
    inference: 150/7968
    inference: 300/7968
    inference: 450/7968
    inference: 600/7968
    inference: 750/7968
    inference: 900/7968
    inference: 1050/7968
    inference: 1200/7968
    inference: 1350/7968
    inference: 1500/7968
    inference: 1650/7968
    inference: 1800/7968
    inference: 1950/7968
    inference: 2100/7968
    inference: 2250/7968
    inference: 2400/7968
    inference: 2550/7968
    inference: 2700/7968
    inference: 2850/7968
    inference: 3000/7968
    inference: 3150/7968
    inference: 3300/7968
    inference: 3450/7968
    inference: 3600/7968
    inference: 3750/7968
    inference: 3900/7968
    inference: 4050/7968
    inference: 4200/7968
    inference: 4350/7968
    inference: 4500/7968
    inference: 4650/7968
    inference: 4800/7968
    inference: 4950/7968
    inference: 5100/7968
    inference: 5250/7968
    inference: 5400/7968
    inference: 5550/7968
    inference: 5700/7968
    inference: 5850/7968
    inference: 6000/7968
    inference: 6150/7968
    inference: 6300/7968
    inference: 6450/7968
    inference: 6600/7968
    inference: 6750/7968
    inference: 6900/7968
    inference: 7050/7968
    inference: 7200/7968
    inference: 7350/7968
    inference: 7500/7968
    inference: 7650/7968
    inference: 7800/7968
    inference: 7950/7968
    inference: 7968/7968
  Поріг=0.600 | Ink=4.93% | RLE=210,726 символів
Збережено /kaggle/working/submission.csv
Збережено submission.csv

========================================================================
submission.csv ГОТОВО! (v22.1-GeneralizationFix)
Ensemble Val F0.5 : 0.6967
Ensemble Threshold: 0.600
========================================================================
  Id                                          Predicted
0  a  2833846 10 2840172 20 2846500 26 2852829 30 28...
1  b  7592 76 7714 67 13921 78 14044 68 20250 80 203...
========================================================================